# ANALISIS GASTOS DEL HOGAR (KAKEBIA)

Elaborado por: Wagner Fernández V.

Especialista en Ciencia de Datos y Analítica

## Importar librerías

In [25]:
import pandas as pd
import os
import numpy as np
import plotly.graph_objects as go 
from statsmodels.tsa.arima.model import ARIMA

## Lectura de datos

In [26]:

# Cargar los datos desde el archivo Excel
df = pd.read_csv("../data/processed/kakebo_merged.csv", sep=";")
df

,MES,MONTO,año
0,MES ENERO,14057968,2025
1,MES FEBRERO,3310580,2025
2,MES MARZO,3072536,2025
3,MES ABRIL,3455498,2025
4,MES MAYO,3385952,2025
5,MES JUNIO,3167049,2025
6,MES JULIO,3337777,2025
7,MES AGOSTO,4465156,2025
8,MES SEPTIEMBRE,5206932,2025
9,MES OCTUBRE,6291127,2025


## Creación de columna relevante para poner el número de mes

In [27]:
# Crear columna "No MES" basada en el nombre del mes
meses = [
    "ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO",
    "JULIO", "AGOSTO", "SEPTIEMBRE", "OCTUBRE", "NOVIEMBRE", "DICIEMBRE"
]
mapa_meses = {mes: i + 1 for i, mes in enumerate(meses)}

df["No MES"] = (
    df["MES"]
    .str.replace("MES ", "", regex=False)
    .str.strip()
    .map(mapa_meses)
)

df



,MES,MONTO,año,No MES
0,MES ENERO,14057968,2025,1
1,MES FEBRERO,3310580,2025,2
2,MES MARZO,3072536,2025,3
3,MES ABRIL,3455498,2025,4
4,MES MAYO,3385952,2025,5
5,MES JUNIO,3167049,2025,6
6,MES JULIO,3337777,2025,7
7,MES AGOSTO,4465156,2025,8
8,MES SEPTIEMBRE,5206932,2025,9
9,MES OCTUBRE,6291127,2025,10


## EDA - Exploratory Data Analysis

### Calculo de la mediana en el DataFrame

In [ ]:
df.mean(numeric_only=True)

TypeError: Could not convert ['MES ENEROMES FEBREROMES MARZOMES ABRILMES MAYOMES JUNIOMES JULIOMES AGOSTOMES SEPTIEMBREMES OCTUBREMES NOVIEMBREMES DICIEMBREMES ENEROMES FEBRERO'] to numeric

### Descripción del DataFrame

In [ ]:
df.describe()

,MONTO,año,No MES
count,1.400000e+01,14.000000,14.000000
mean,5.380020e+06,2025.142857,5.785714
std,2.985471e+06,0.363137,3.786181
min,3.072536e+06,2025.000000,1.000000
25%,3.349821e+06,2025.000000,2.250000
50%,4.716576e+06,2025.000000,5.500000
75%,5.982803e+06,2025.000000,8.750000
max,1.405797e+07,2026.000000,12.000000


## Desarrollo y Aplicación del modelo ARIMA

In [ ]:
# Preparar la serie temporal (solo datos reales)
serie_arima = df['MONTO'].values

# Ajustar modelo ARIMA (5,1,0) basado en tu análisis previo
modelo_arima_final = ARIMA(serie_arima, order=(5, 1, 0))
resultado_arima_final = modelo_arima_final.fit()

# Predecir el siguiente mes (Marzo 2026)
prediccion_siguiente_mes = resultado_arima_final.forecast(steps=1)
print(f"Predicción para Marzo 2026: ${prediccion_siguiente_mes[0]:,.2f}")

# Crear DataFrame con histórico y predicción
df_historico = df[['MES', 'MONTO', 'año', 'No MES']].copy()
df_historico['TIPO'] = 'REAL'

# Agregar predicción del siguiente mes
fila_prediccion = pd.DataFrame({
    'MES': ['MES MARZO'],
    'MONTO': [prediccion_siguiente_mes[0]],
    'año': [2026],
    'No MES': [3],
    'TIPO': ['PREDICCION']
})

df_completo = pd.concat([df_historico, fila_prediccion], ignore_index=True)

# Crear etiquetas para el eje X
df_completo['MES_ANO'] = df_completo['MES'] + ' ' + df_completo['año'].astype(str)


c:\Users\WAGNER FERNÁNDEZ\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning:

Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.



Predicción para Marzo 2026: $6,576,764.80


## Graficar el modelo ARIMA con Plotly

In [ ]:
# Graficar resultados
fig_arima = go.Figure()

# Datos reales
df_reales = df_completo[df_completo['TIPO'] == 'REAL']
fig_arima.add_trace(go.Scatter(
    x=df_reales['MES_ANO'],
    y=df_reales['MONTO'],
    mode='lines+markers',
    name='REAL',
    line=dict(color='blue', width=2),
    marker=dict(size=8)
))

# Predicción
df_pred_arima = df_completo[df_completo['TIPO'] == 'PREDICCION']
fig_arima.add_trace(go.Scatter(

    y=df_pred_arima['MONTO'],
    mode='markers',
    name='PREDICCION ARIMA',
    marker=dict(color='red', size=12, symbol='star')
))

fig_arima.update_layout(
    title='Predicciones KAKEBIA',
    title_x=0.5,
    xaxis_title='Mes y año',
    yaxis_title='Monto en pesos colombianos (COP)',
    hovermode='x unified',
    template='plotly_white'
)

fig_arima.show()

# Formatear MONTO para exportación
df_completo['MONTO_FORMATEADO'] = df_completo['MONTO'].apply(lambda x: f"{x:,.0f}".replace(',', '.'))

## Exportación del Archivo para la Analítica

In [ ]:
# Exportar a CSV
df_export = df_completo[['MES', 'MONTO_FORMATEADO', 'año', 'TIPO']].copy()
df_export.columns = ['MES', 'MONTO', 'AÑO', 'TIPO_DATO']
df_export.to_csv("../data/processed/kakebo_pred2.csv", sep=";", index=False)

print("\n✓ Archivo exportado: kakebo_pred2.csv")
print(f"\nResumen del modelo ARIMA:")
print(resultado_arima_final.summary())


✓ Archivo exportado: kakebo_pred2.csv

Resumen del modelo ARIMA:
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                   14
Model:                 ARIMA(5, 1, 0)   Log Likelihood                -210.517
Date:               jue, 19 feb. 2026   AIC                            433.034
Time:                        15:24:10   BIC                            436.424
Sample:                             0   HQIC                           432.338
                                 - 14                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.7079      0.086     -8.215      0.000      -0.877      -0.539
ar.L2         -0.3972      0.151     -2.632      0.008      -0.69